# Bronze Layer, Products Data Ingestion
**Source:** MongoDB (`mongodb://host.docker.internal:27017/products_1.products_1`)  
**Target:** HDFS Delta Table `/delta/bronze/products`  
**Pattern:** Full Load with Ingestion Timestamp  
**Layer:** Bronze (Raw Ingestion)

---

In [3]:
import os, sys, logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "mongo_uri"    : "mongodb://host.docker.internal:27017/products_1.products_1",  
    "bronze_path"  : "hdfs://master:8020/delta/bronze/products",
    "database"     : "bronze",
    "table_name"   : "bronze_products",
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Bronze_Products_Load")
        .master("spark://master:7077")
        .config("spark.jars.packages",
                "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0,"
                "io.delta:delta-spark_2.12:3.2.1")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .config("spark.databricks.delta.schema.autoMerge.enabled", "true")   
        .config("spark.mongodb.read.connection.uri", CONFIG["mongo_uri"])
        .enableHiveSupport()
        .getOrCreate()
    )

def run():
    logger.info("Products Bronze pipeline started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    # Read from MongoDB
    logger.info(f"Reading from MongoDB: {CONFIG['mongo_uri']}")
    df_raw = spark.read.format("mongodb").load()
    count_raw = df_raw.count()
    logger.info(f"Extracted {count_raw} rows from MongoDB.")

    if count_raw == 0:
        logger.warning("No data extracted. Exiting without writing.")
        return

    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CONFIG['database']}")
    spark.sql(f"DROP TABLE IF EXISTS {CONFIG['database']}.{CONFIG['table_name']}")

    logger.info(f"Writing Delta to {CONFIG['bronze_path']}...")
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("delta.columnMapping.mode", "name") \
        .save(CONFIG["bronze_path"])

    spark.sql(f"""
        CREATE TABLE {CONFIG['database']}.{CONFIG['table_name']}
        USING DELTA
        LOCATION '{CONFIG['bronze_path']}'
    """)

    count = spark.read.format("delta").load(CONFIG["bronze_path"]).count()
    logger.info(f"SUCCESS: {count} rows in Delta table.")
    spark.read.format("delta").load(CONFIG["bronze_path"]).show(5, truncate=False)

if __name__ == "__main__":
    run()

2026-03-18 19:38:48,558 [INFO] Products Bronze pipeline started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cfd86381-2afd-4fe0-8c53-d7ae8c2a1ff2;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.12/10.3.0/mongo-spark-connector_2.12-10.3.0.jar ...
	[SUCCESSFUL ] org.mongodb.spa

+------------------------+------------+-----------+---------+---------+------+----------+------+--------------+-----------------------+
|_id                     |brand       |category   |is_active|name     |price |product_id|rating|stock_quantity|ingestion_timestamp    |
+------------------------+------------+-----------+---------+---------+------+----------+------+--------------+-----------------------+
|69b2dbcde706b09c2944152e|BeautyGlow  |Toys       |true     |Product 1|995.73|1         |3.5   |989           |2026-03-18 19:39:21.789|
|69b2dbcde706b09c2944152f|GardenMaster|Garden     |true     |Product 2|497.76|2         |3.8   |495           |2026-03-18 19:39:21.789|
|69b2dbcde706b09c29441530|BeautyGlow  |Electronics|true     |Product 3|331.63|3         |4.6   |10            |2026-03-18 19:39:21.789|
|69b2dbcde706b09c29441531|TechPro     |Beauty     |false    |Product 4|798.83|4         |4.7   |683           |2026-03-18 19:39:21.789|
|69b2dbcde706b09c29441532|HomeSmart   |Automotiv